# Week 3: Sentiment Analysis

## Learning Objectives
- Build rule-based and ML-based sentiment classifiers
- Use VADER for social media text
- Train a Naive Bayes text classifier
- Evaluate sentiment models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

try:
    import nltk
    nltk.download('vader_lexicon', quiet=True)
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    VADER = True
except ImportError:
    VADER = False

sns.set_theme(style='whitegrid')

## 1. VADER Rule-Based Sentiment

In [ ]:
if VADER:
    sia = SentimentIntensityAnalyzer()
    test_sentences = [
        'I absolutely love this product! It is amazing!',
        'This is the worst experience I have ever had.',
        'The movie was okay, nothing special.',
        'Fantastic service, highly recommended!',
        'Terrible quality, complete waste of money.'
    ]
    for s in test_sentences:
        scores = sia.polarity_scores(s)
        label = 'Positive' if scores['compound'] > 0.05 else ('Negative' if scores['compound'] < -0.05 else 'Neutral')
        print(f'{label:8s} ({scores["compound"]:+.3f}): {s[:50]}')
else:
    print('NLTK not installed.')

## 2. Movie Review Sentiment with Sklearn

In [ ]:
from sklearn.datasets import load_files
import os

# Create a simple synthetic dataset if the real one is unavailable
positive = [
    'great movie loved every minute of it wonderful acting',
    'excellent film highly recommend amazing story',
    'brilliant performance outstanding cinematography',
    'fantastic thriller kept me on the edge of my seat',
    'wonderful experience the best movie this year'
] * 40

negative = [
    'terrible movie waste of time horrible acting',
    'worst film ever made completely boring',
    'awful production poor script nothing to recommend',
    'disappointing sequel nothing worked in this film',
    'dreadful experience I walked out halfway through'
] * 40

texts = positive + negative
labels = [1]*len(positive) + [0]*len(negative)

X_train, X_test, y_train, y_test = train_test_split(texts, labels, stratify=labels, test_size=0.2, random_state=42)

for name, clf in [
    ('Naive Bayes',  Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', MultinomialNB())])),
    ('Logistic Reg', Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=1000))]))
]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f'{name:15s}: {acc:.4f}')

## 3. Visualising Most Predictive Words

In [ ]:
pipeline = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=1000))])
pipeline.fit(X_train, y_train)

vocab = pipeline['tfidf'].get_feature_names_out()
coefs = pipeline['clf'].coef_[0]

top_pos = pd.Series(coefs, index=vocab).nlargest(10)
top_neg = pd.Series(coefs, index=vocab).nsmallest(10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
top_pos.plot.barh(ax=axes[0], color='green', alpha=0.7)
axes[0].set_title('Most Positive Words')
top_neg.abs().plot.barh(ax=axes[1], color='red', alpha=0.7)
axes[1].set_title('Most Negative Words')
plt.tight_layout(); plt.show()

## Exercise

1. Collect 50 tweets (or use a public dataset) and classify their sentiment with VADER
2. Compare VADER vs Logistic Regression on the same test set
3. Add emoji handling to improve VADER accuracy

In [ ]:
# Your code here


## Summary

- ✓ VADER rule-based sentiment analysis
- ✓ Naive Bayes and Logistic Regression classifiers
- ✓ TF-IDF feature extraction for sentiment
- ✓ Visualising predictive words

## Next Week
Sequence-to-Sequence models!